# Backtest EMACrossStopEntry

Run the EMA regime-based strategy with breakout confirmation entry (manual trigger check + MARKET order) and trailing stop exit on a simulated exchange venue.

**Strategy logic:**
- **Entry:** Breakout confirmation — BUY trigger at bar.high + (ATR × offset), SELL at bar.low - (ATR × offset). Checked on the NEXT bar; enters via MARKET order if confirmed.
- **Exit:** Trailing stop market order (ATR-based offset), submitted automatically on entry fill
- Only enters when flat + EMA regime aligned. Requires next bar to break beyond current bar's high/low before entering.
- After trailing stop exit, re-enters on the next bar if still in EMA regime (regime-based)
- **No EMA reversal handling** — trailing stop is the sole exit mechanism

**Key difference from EMACrossTrailingStop:**
EMACrossTrailingStop uses immediate MARKET entry. EMACrossStopEntry requires the next bar
to break beyond the previous bar's high/low — filtering false signals via breakout confirmation.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — EMA period grid, then entry offset × trailing multiplier sensitivity

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.ema_cross_stop_entry import EMACrossStopEntry, EMACrossStopEntryConfig

from charts import plot_ema_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = False

# EMA params
FAST_EMA = 10
SLOW_EMA = 20

# ATR + trailing stop params
ATR_PERIOD        = 20
TRAILING_ATR_MULT = 3.0

# Entry breakout params
ENTRY_OFFSET_ATR = 0.25    # Breakout trigger: bar.high + (ATR × 0.25)

# Sweep grids — EMA periods
FAST_PERIODS = sorted(set([5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50] + [FAST_EMA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_EMA]))

# Sweep grids — entry offset and trailing multiplier
ENTRY_OFFSET_ATR_GRID = [0.1, 0.25, 0.5, 0.75, 1.0]
TRAILING_ATR_MULTS    = [1.5, 2.0, 3.0, 4.0, 5.0]

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"EMACrossStopEntry_{INSTRUMENT_ID}_{BAR_INTERVAL}_f{FAST_EMA}_s{SLOW_EMA}_eo{ENTRY_OFFSET_ATR}_trail{TRAILING_ATR_MULT}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL, log_level="INFO")
print("Engine configured.")

In [ ]:
# ── Cell 4: Add EMACrossStopEntry strategy + run ────────────
config = EMACrossStopEntryConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
    atr_period=ATR_PERIOD,
    trailing_atr_multiple=TRAILING_ATR_MULT,
    entry_offset_atr=ENTRY_OFFSET_ATR,
)
strategy = EMACrossStopEntry(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ────────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ──────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"EMACrossStopEntry({FAST_EMA}/{SLOW_EMA}, eo={ENTRY_OFFSET_ATR}, trail={TRAILING_ATR_MULT}x) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with EMA overlay + trade markers ────────────
#
# Reuses plot_ema_cross() — same EMA indicators, trade markers show
# entries and trailing stop fills.

fig = plot_ema_cross(
    bars, fills_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ────────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"EMACrossStopEntry({FAST_EMA}/{SLOW_EMA}, ATR={ATR_PERIOD}, eo={ENTRY_OFFSET_ATR}, trail={TRAILING_ATR_MULT}×)  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ─────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep — EMA fast/slow periods ─────────────
#
# Grid search over EMA period combinations.
# Entry offset and trailing params held at defaults.

def stop_entry_factory(eng, params):
    cfg = EMACrossStopEntryConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        fast_ema_period=params.get("fast", FAST_EMA),
        slow_ema_period=params.get("slow", SLOW_EMA),
        atr_period=params.get("atr_period", ATR_PERIOD),
        trailing_atr_multiple=params.get("trail_mult", TRAILING_ATR_MULT),
        entry_offset_atr=params.get("entry_offset_atr", ENTRY_OFFSET_ATR),
    )
    eng.add_strategy(EMACrossStopEntry(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=stop_entry_factory,
    strategy_name="EMACrossStopEntry",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap — EMA fast vs slow ────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="Slow EMA Period", col_label="Fast EMA Period",
    title=f"Total PnL ({CURRENCY}) by EMA Parameters",
)

In [ ]:
# ── Cell 13: Entry offset × trailing multiplier sensitivity sweep ───
#
# Fix EMA at the best combo from the previous sweep.
# Vary entry offset ATR fraction and trailing ATR multiplier together —
# tests the breakout threshold vs stop tightness tradeoff.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
print(f"Best EMA params: fast={best_fast}, slow={best_slow} (PnL={best_row['total_pnl']:,.2f})")

combos = [{"entry_offset_atr": eo, "trail_mult": tm, "fast": best_fast, "slow": best_slow}
          for eo in ENTRY_OFFSET_ATR_GRID for tm in TRAILING_ATR_MULTS]

sens_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=stop_entry_factory,
    strategy_name="EMACrossStopEntry-sensitivity",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

plot_pnl_heatmap(
    sens_df, row_col="entry_offset_atr", col_col="trail_mult",
    row_label="Entry Offset (ATR fraction)", col_label="Trailing ATR Multiple",
    title=f"Total PnL ({CURRENCY}) by Entry Offset vs Trailing Multiple — EMA({best_fast}/{best_slow})",
)

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ────────────────────

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Save notebook snapshot ────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_ema_cross_stop_entry.ipynb", RESULT_NAME)
#save_notebook_html("backtest_ema_cross_stop_entry.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 16: Cleanup ──────────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")